# Lab4.1 — Codificação de Áudio com Perdas (v2)
**Parte A | 10–30 min.** Recupere gravações WAV do Lab 3, selecione um trecho curto e codifique em MP3, AAC e Ogg Vorbis, em duas taxas por codec. **Não faça o teste perceptivo neste notebook**: utilize o Lab4.4 com o grupo parceiro.
**Objetivos:** codificar gravações próprias; medir bitrate/tamanho; comparar condições de codificação; preparar arquivos para avaliação cega.
**Cuidados:** não use gravações contendo nomes completos, números de matrícula ou dados pessoais. Selecione sinal sem clipping, com 10–15 s, e mantenha uma cópia WAV de referência.

In [ ]:
from pathlib import Path
import os
from google.colab import drive
drive.mount('/content/drive')
GRUPO = "G4"  # ALTERAR: G1, G2, ..., G6
RAIZ = Path('/content/drive/MyDrive/Lab4_CSM')
PASTA = RAIZ / GRUPO
PASTA.mkdir(parents=True, exist_ok=True)
print("Pasta:", PASTA)

## 1. Selecione a gravação WAV do Lab 3
Coloque o arquivo WAV em `Lab4_CSM/G4/originais/` e edite `NOME_WAV`. A conversão para PCM 16-bit é usada apenas para padronizar a referência do teste.

In [ ]:
import subprocess, shutil, json, pandas as pd
ORIG = PASTA / 'originais'; ORIG.mkdir(exist_ok=True)
NOME_WAV = 'estação1.wav'  # ALTERAR
entrada = ORIG / NOME_WAV
assert entrada.exists(), f'Arquivo não encontrado: {entrada}'
assert shutil.which('ffmpeg'), 'FFmpeg indisponível'
referencia = PASTA / 'audio' / 'referencia.wav'
referencia.parent.mkdir(exist_ok=True)
def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode: raise RuntimeError(p.stderr[-1800:])
    return p
run(['ffmpeg','-y','-i',str(entrada),'-t','12','-ar','44100','-ac','2','-c:a','pcm_s16le',str(referencia)])
print('Referência:', referencia)

## 2. Codifique seis versões
Duas taxas nominais por codec: 48 e 128 kbit/s. As taxas efetivas e tamanhos serão medidos. AAC é armazenado em `.m4a`, MP3 em `.mp3` e Vorbis em `.ogg`.

In [ ]:
SAIDA = PASTA / 'audio' / 'codificados'; SAIDA.mkdir(parents=True,exist_ok=True)
config = [('MP3','libmp3lame','.mp3'),('AAC','aac','.m4a'),('Vorbis','libvorbis','.ogg')]
registros=[]
for codec, encoder, ext in config:
    for kbps in (48,128):
        dest=SAIDA/f'{codec.lower()}_{kbps}k{ext}'
        run(['ffmpeg','-y','-i',str(referencia),'-vn','-c:a',encoder,'-b:a',f'{kbps}k',str(dest)])
        probe=json.loads(run(['ffprobe','-v','error','-show_entries','format=duration,bit_rate,size','-of','json',str(dest)]).stdout)
        registros.append({'grupo':GRUPO,'codec':codec,'kbps_nominal':kbps,'arquivo':dest.name,
                          'bytes':dest.stat().st_size,'kbps_efetivo':round(float(probe['format']['bit_rate'])/1000,1)})
tabela=pd.DataFrame(registros)
display(tabela)
tabela.to_csv(PASTA/'audio'/'medidas_audio.csv',index=False)

## 3. Conferência e transferência
Abra **Lab4.4** para gerar os estímulos anônimos. Compartilhe com o grupo parceiro apenas a pasta de avaliação (não o gabarito). **G1 ↔ G2; G3 ↔ G4; G5 ↔ G6.** Para explorar fala × canto, outro integrante pode repetir o processo com outro WAV.